In [1]:
import torch

graph = torch.load("./dataset/graphs/action-recognition-in-videos-on-something.pt")
print(graph)

/tmp/ipykernel_69985/2372515897.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  graph = torch.load("./dataset/graphs/action-recognition-in-videos-on-something.pt")


Data(x=[0], edge_index=[2, 1565], arxiv_id=[876], content=[876], abstract=[876], title=[876], desc='Merged graph statistics:
Original total nodes before merging: 1792
Number of nodes: 876
Number of edges: 1565

Root papers to build up the graph:
2311.15769v1: Side4Video: Spatial-Temporal Side Network for Memory-Efficient Image-to-Video Transfer Learning
2112.09133v2: Masked Feature Prediction for Self-Supervised Visual Pre-Training
2203.12602v3: VideoMAE: Masked Autoencoders are Data-Efficient Learners for Self-Supervised Video Pre-Training
2207.11660v1: MAR: Masked Autoencoders for Efficient Action Recognition
2212.04500v2: Masked Video Distillation: Rethinking Masked Feature Modeling for Self-supervised Video Representation Learning
2303.12001v2: ViC-MAE: Self-Supervised Representation Learning from Images and Video with Contrastive Masked Autoencoders
2212.03191v2: InternVideo: General Video Foundation Models via Generative and Discriminative Learning
2306.00989v1: Hiera: A Hierarch

In [2]:
from src.utils.citation_graph import to_hierarchical

hier_version = to_hierarchical(graph, 1000)
print(hier_version)

Data(edge_index=[2, 35578], node_text=[31048], root_labels=[31048])


In [3]:
root_indices = torch.where(hier_version.root_labels == 1)[0]
print(root_indices.shape)

torch.Size([847])


In [12]:
def hier_graphs(merged_paper_graph: Data, citation_graph: Data) -> Data:
    """
    Merge hierarchical and citation graphs by adding edges between matching root nodes,
            
    Returns:
        Data: Updated hierarchical graph with new edges from citation network
    """

    root_indices = torch.where(merged_paper_graph.root_labels == 1)[0]

    chunks_to_idx = {merged_paper_graph.node_text[idx]: idx for idx in root_indices}
    cit_title_to_idx = {title: idx for idx, title in enumerate(citation_graph.title)}
    cit_idx_to_title = {idx: title for idx, title in enumerate(citation_graph.title)}

    new_edges = []

    for hier_idx in root_indices:
        hier_title = merged_paper_graph.node_text[hier_idx]

        if hier_title in cit_title_to_idx:
            cit_idx = cit_title_to_idx[hier_title]

            # Get all citations from this paper in citation graph
            citation_edges = citation_graph.edge_index
            source_mask = citation_edges[0] == cit_idx
            cited_indices = citation_edges[1][source_mask]

            for cited_idx in cited_indices:
                cited_title = cit_idx_to_title[cited_idx.item()]
                
                # If the cited paper exists as a root node in hierarchical graph
                if cited_title in chunks_to_idx:
                    cited_hier_idx = chunks_to_idx[cited_title]
                    new_edges.append((hier_idx, cited_hier_idx))

            # Also get papers that cite this paper
            target_mask = citation_edges[1] == cit_idx
            citing_indices = citation_edges[0][target_mask]
            
            # For each citing paper
            for citing_idx in citing_indices:
                citing_title = cit_idx_to_title[citing_idx.item()]
                
                # If the citing paper exists as a root node in hierarchical graph
                if citing_title in chunks_to_idx:
                    citing_hier_idx = chunks_to_idx[citing_title]
                    new_edges.append((citing_hier_idx, hier_idx))

    if new_edges:
        # Convert new edges to tensor
        new_edges_tensor = torch.tensor(new_edges, dtype=torch.long).t()
        
        # Combine with existing edges
        updated_edge_index = torch.cat([merged_paper_graph.edge_index, new_edges_tensor], dim=1)
        
        # Create updated graph
        updated_graph = Data(
            edge_index=updated_edge_index,
            node_text=merged_paper_graph.node_text,
            root_labels=merged_paper_graph.root_labels
        )
    else:
        updated_graph = merged_paper_graph
    
    return updated_graph

    
hier_graph = hier_graphs(merged_graph, graph)
print(hier_graph)



Data(edge_index=[2, 88], node_text=[76], root_labels=[76])


In [16]:
import re
import itertools
from collections import defaultdict
import torch
from torch_geometric.data import Data
import spacy

nlp = spacy.load("en_core_web_sm")

def split_into_chunks(text, chunk_size=100):
    words = text.split()
    chunks = [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]
    return chunks

def build_graph_from_tex_content(content, main_title="Main Title"):
    node_list = []
    edges = []
    section_refs = defaultdict(list)
    section_stack = []  # Stack to manage the hierarchy of sections

    # Add the root node
    root_node = len(node_list)
    node_list.append({"title": main_title, "content": "", "type": "root"})

    # Regex pattern to match any level of section (\section, \subsection, \subsubsection, etc.)
    pattern = r'(\\(sub)*section\{.*?\})'
    parts = re.split(pattern, content)

    current_parent_node = root_node  # Start with root as the initial parent
    for i in range(len(parts)):
        if isinstance(parts[i], str) and re.match(r'\\(sub)*section', parts[i]):
            # Calculate depth based on the number of "sub" prefixes
            depth = parts[i].count("sub")
            title_match = re.match(r'\\(sub)*section\{(.*?)\}', parts[i])
            if title_match:
                section_title = title_match.group(2)

                # Remove nodes from stack that are at the same or deeper level
                while section_stack and section_stack[-1][1] >= depth:
                    section_stack.pop()

                # Create a new intermediate node for this section
                section_node = len(node_list)
                node_list.append({"title": section_title, "content": "", "type": f"section_level_{depth}"})

                # Determine the parent node
                if section_stack:
                    parent_node = section_stack[-1][0]
                else:
                    parent_node = root_node  # Connect top-level sections to the root

                # Connect the current section to its parent
                edges.append((parent_node, section_node))

                # Push the current section onto the stack with its depth level
                section_stack.append((section_node, depth))

                # If there is content after the section title, split it into chunks
                if i + 1 < len(parts) and isinstance(parts[i + 1], str) and parts[i + 1]:
                    section_content = parts[i + 1]
                    chunks = split_into_chunks(section_content)
 
                    # Create leaf nodes for each chunk and connect to the current section
                    for chunk in chunks:
                        leaf_node = len(node_list)
                        node_list.append({"title": "", "content": chunk, "type": "chunk"})
                        edges.append((section_node, leaf_node))

                        # Check for references in each chunk
                        refs = re.findall(r'\\ref\{(.*?)\}', chunk)
                        for ref in refs:
                            section_refs[ref].append(leaf_node)
        else:
            # Content outside of recognized sections
            continue

    return node_list, edges, section_refs



def extract_entities(text):
    doc = nlp(text)
    entities = [ent.text for ent in doc.ents]
    return entities

def compute_edges_with_entities(node_list):
    edge_list = []
    edge_attr_list = []
    node_entities = [extract_entities(node['content']) for node in node_list]

    for (i, entities_i), (j, entities_j) in itertools.combinations(enumerate(node_entities), 2):
        shared_entities = set(entities_i).intersection(set(entities_j)) 
        shared_count = len(shared_entities)
        if shared_count > 0:
            edge_list.append([i, j])
            edge_list.append([j, i])
            edge_attr_list.append(shared_count)
            edge_attr_list.append(shared_count)

    return edge_list, edge_attr_list

def create_pytorch_graph(node_list, initial_edges, section_refs):
    edge_list, edge_attr_list = compute_edges_with_entities(node_list)
    edge_list.extend(initial_edges)

    # Add cross-reference edges based on section_refs
    for ref, leaf_nodes in section_refs.items():
        for leaf_node in leaf_nodes:
            ref_index = next((i for i, n in enumerate(node_list) if n["title"] == ref), None)
            if ref_index is not None:
                edge_list.append((leaf_node, ref_index))
                edge_list.append((ref_index, leaf_node))

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    num_nodes = len(node_list)
    x = torch.arange(num_nodes, dtype=torch.float32).unsqueeze(1)
    edge_attr = torch.tensor(edge_attr_list, dtype=torch.float32).unsqueeze(1)
    graph_data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

    return graph_data

# Example usage
if __name__ == "__main__":
    # Example LaTeX content with multiple nested sections
    tex_content = r"""
    \section{Introduction}
    This section provides an overview. As discussed in \ref{related_work}.
    \subsection{Background}
    Background content here.
    \subsubsection{Detailed Background}
    More specific background information.
    \subsection{Motivation}
    Motivation content. See also \ref{methods}.

    \section{Related Work}
    Related work discussed here.

    \section{Methods}
    Detailed methods are described here.
    """

    # Build graph
    node_list, initial_edges, section_refs = build_graph_from_tex_content(tex_content)
    graph = create_pytorch_graph(node_list, initial_edges, section_refs)

    # Output graph data
    print("Node List:", node_list)
    print(f"Edge Index (References):\n{graph.edge_index}")
    print(f"Edge Attributes (Shared Entities):\n{graph.edge_attr}")


Node List: [{'title': 'Sample Paper Title', 'content': 'Sample Paper Title', 'type': 'root'}, {'title': 'Introduction', 'content': 'Introduction', 'type': 'section_level_0'}, {'title': 'Background', 'content': 'Background', 'type': 'section_level_1'}, {'title': '', 'content': 'sub', 'type': 'chunk'}, {'title': 'Detailed Background', 'content': 'Detailed Background', 'type': 'section_level_2'}, {'title': '', 'content': 'sub', 'type': 'chunk'}, {'title': 'Motivation', 'content': 'Motivation', 'type': 'section_level_1'}, {'title': '', 'content': 'sub', 'type': 'chunk'}, {'title': 'Related Work', 'content': 'Related Work', 'type': 'section_level_0'}, {'title': 'Methods', 'content': 'Methods', 'type': 'section_level_0'}]
Edge Index (References):
tensor([[0, 1, 2, 2, 4, 1, 6, 0, 0],
        [1, 2, 3, 4, 5, 6, 7, 8, 9]])
Node Text Attributes:
['Sample Paper Title', 'Introduction', 'Background', 'sub', 'Detailed Background', 'sub', 'Motivation', 'sub', 'Related Work', 'Methods']


In [22]:
import re
import itertools
from collections import defaultdict
import torch
from torch_geometric.data import Data
import spacy

nlp = spacy.load("en_core_web_sm")

def split_into_chunks(text, max_tokens=100):
    words = text.split()
    chunks = [' '.join(words[i:i + max_tokens]) for i in range(0, len(words), max_tokens)]
    return chunks

def build_graph_from_tex_content(content, main_title="Main Title", chunk_size=100):
    node_list = []
    edges = []
    node_text = []  # Store text attributes for each node
    section_refs = defaultdict(list)
    section_stack = []  # Stack to manage the hierarchy of sections

    # Add the root node with the main title
    root_node = len(node_list)
    node_list.append({"title": main_title, "content": main_title, "type": "root"})
    node_text.append(main_title)  # Root node text attribute

    # Regex pattern to match any level of section (\section, \subsection, \subsubsection, etc.)
    pattern = r'(\\(sub)*section\{.*?\})'
    parts = re.split(pattern, content)

    current_parent_node = root_node  # Start with root as the initial parent
    for i in range(len(parts)):
        if isinstance(parts[i], str) and re.match(r'\\(sub)*section', parts[i]):
            # Calculate depth based on the number of "sub" prefixes
            depth = parts[i].count("sub")
            title_match = re.match(r'\\(sub)*section\{(.*?)\}', parts[i])
            if title_match:
                section_title = title_match.group(2)

                # Remove nodes from stack that are at the same or deeper level
                while section_stack and section_stack[-1][1] >= depth:
                    section_stack.pop()

                # Create a new intermediate node for this section with title as the text attribute
                section_node = len(node_list)
                node_list.append({"title": section_title, "content": section_title, "type": f"section_level_{depth}"})
                node_text.append(section_title)  # Add section title to node_text

                # Determine the parent node
                if section_stack:
                    parent_node = section_stack[-1][0]
                else:
                    parent_node = root_node  # Connect top-level sections to the root

                # Connect the current section to its parent
                edges.append((parent_node, section_node))

                # Push the current section onto the stack with its depth level
                section_stack.append((section_node, depth))

                # Determine if this is a final section without further subsections
                is_final_section = (
                    (i + 2 >= len(parts)) or  # No further sections
                    not re.match(r'\\(sub)*section', parts[i + 2]) or  # No more sections in parts
                    parts[i + 2].count("sub") <= depth  # Next section is a higher or same level
                )

                # If this is a final section, split its content into chunks as leaf nodes
                if is_final_section and i + 1 < len(parts) and isinstance(parts[i + 1], str) and parts[i + 1]:
                    section_content = parts[i + 1]
                    chunks = split_into_chunks(section_content, max_tokens=chunk_size)

                    # Create leaf nodes for each chunk and connect to the current section
                    for chunk in chunks:
                        leaf_node = len(node_list)
                        node_list.append({"title": "", "content": chunk, "type": "chunk"})
                        edges.append((section_node, leaf_node))
                        node_text.append(chunk)  # Add chunk content to node_text

                        # Check for references in each chunk
                        refs = re.findall(r'\\ref\{(.*?)\}', chunk)
                        for ref in refs:
                            section_refs[ref].append(leaf_node)
        else:
            # Content outside of recognized sections
            continue

    return node_list, edges, section_refs, node_text

def extract_entities(text):
    doc = nlp(text)
    entities = [ent.text for ent in doc.ents]
    return entities

def compute_edges_with_entities(node_list):
    edge_list = []
    edge_attr_list = []
    node_entities = [extract_entities(node['content']) for node in node_list]

    for (i, entities_i), (j, entities_j) in itertools.combinations(enumerate(node_entities), 2):
        shared_entities = set(entities_i).intersection(set(entities_j)) 
        shared_count = len(shared_entities)
        if shared_count > 0:
            edge_list.append([i, j])
            edge_list.append([j, i])
            edge_attr_list.append(shared_count)
            edge_attr_list.append(shared_count)

    return edge_list, edge_attr_list

def create_pytorch_graph(node_list, initial_edges, section_refs, node_text):
    edge_list, edge_attr_list = compute_edges_with_entities(node_list)
    edge_list.extend(initial_edges)

    # Add cross-reference edges based on section_refs
    for ref, leaf_nodes in section_refs.items():
        for leaf_node in leaf_nodes:
            ref_index = next((i for i, n in enumerate(node_list) if n["title"] == ref), None)
            if ref_index is not None:
                edge_list.append((leaf_node, ref_index))
                edge_list.append((ref_index, leaf_node))

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    num_nodes = len(node_list)
    x = torch.arange(num_nodes, dtype=torch.float32).unsqueeze(1)  # Node feature matrix as indices

    graph_data = Data(x=x, edge_index=edge_index, node_text=node_text)  # Adding node_text as attribute

    return graph_data

# Example usage
if __name__ == "__main__":
    # Example LaTeX content with multiple nested sections
    tex_content = r"""
    \section{Introduction}
    This section provides an overview. As discussed in \ref{related_work}.
    \subsection{Background}
    Background content here.
    \subsubsection{Detailed Background}
    More specific background information.
    \subsection{Motivation}
    Motivation content. See also \ref{methods}.

    \section{Related Work}
    Related work discussed here.

    \section{Methods}
    Detailed methods are described here.
    """

    # Build graph
    node_list, initial_edges, section_refs, node_text = build_graph_from_tex_content(tex_content, main_title="Sample Paper Title", chunk_size=50)
    graph = create_pytorch_graph(node_list, initial_edges, section_refs, node_text)

    # Output graph data
    print("Node List:", node_list)
    print(f"Edge Index (References):\n{graph.edge_index}")
    print(f"Node Text Attributes:\n{graph.node_text}")


Node List: [{'title': 'Sample Paper Title', 'content': 'Sample Paper Title', 'type': 'root'}, {'title': 'Introduction', 'content': 'Introduction', 'type': 'section_level_0'}, {'title': 'Background', 'content': 'Background', 'type': 'section_level_1'}, {'title': '', 'content': 'sub', 'type': 'chunk'}, {'title': 'Detailed Background', 'content': 'Detailed Background', 'type': 'section_level_2'}, {'title': '', 'content': 'sub', 'type': 'chunk'}, {'title': 'Motivation', 'content': 'Motivation', 'type': 'section_level_1'}, {'title': '', 'content': 'sub', 'type': 'chunk'}, {'title': 'Related Work', 'content': 'Related Work', 'type': 'section_level_0'}, {'title': 'Methods', 'content': 'Methods', 'type': 'section_level_0'}]
Edge Index (References):
tensor([[0, 1, 2, 2, 4, 1, 6, 0, 0],
        [1, 2, 3, 4, 5, 6, 7, 8, 9]])
Node Text Attributes:
['Sample Paper Title', 'Introduction', 'Background', 'sub', 'Detailed Background', 'sub', 'Motivation', 'sub', 'Related Work', 'Methods']


In [32]:
import re

def parse_sections(content):
    # Pattern to match any section command, capturing the number of "sub" prefixes and section title
    pattern = r'(\\(sub)*section\{(.*?)\})'
    sections = []
    
    # Find all section commands and their titles
    matches = list(re.finditer(pattern, content))
    for i, match in enumerate(matches):
        section_command = match.group(1)  # Full section command
        title = match.group(3)  # Extract the section title
        start_index = match.start(1)  # Position in the content where the section starts
        end_index = matches[i + 1].start(1) if i + 1 < len(matches) else len(content)
        
        # Determine if this is a leaf node by checking if the next section has a higher or same level
        is_leaf = True  # Default to True (assume it's a leaf)
        if i + 1 < len(matches):
            next_command = matches[i + 1].group(1)
            # It's a leaf only if the next section is of a higher or equal hierarchy
            is_leaf = next_command.count("sub") <= section_command.count("sub")
        
        sections.append({
            "title": title,
            "start_index": start_index,
            "end_index": end_index,
            "is_leaf": is_leaf
        })
    
    return sections

# Example usage
if __name__ == "__main__":
    tex_content = r"""
    \section{Introduction}
    This section provides an overview. As discussed in \ref{related_work}.
    \subsection{Background}
    Background content here.
    \subsubsection{Detailed Background}
    More specific background information.
    \subsubsection{Preliminaries}
    More specific details
    \subsection{Motivation}
    Motivation content. See also \ref{methods}.

    \section{Related Work}
    Related work discussed here.

    \section{Methods}
    Detailed methods are described here.
    """

    sections = parse_sections(tex_content)
    for sec in sections:
        content_type = "Leaf" if sec["is_leaf"] else "Non-leaf"
        print(f"Title: {sec['title']}, Start Index: {sec['start_index']}, End Index: {sec['end_index']}, Type: {content_type}")


Title: Introduction, Start Index: 5, End Index: 107, Type: Non-leaf
Title: Background, Start Index: 107, End Index: 164, Type: Non-leaf
Title: Detailed Background, Start Index: 164, End Index: 246, Type: Leaf
Title: Preliminaries, Start Index: 246, End Index: 306, Type: Leaf
Title: Motivation, Start Index: 306, End Index: 383, Type: Leaf
Title: Related Work, Start Index: 383, End Index: 444, Type: Leaf
Title: Methods, Start Index: 444, End Index: 507, Type: Leaf


In [61]:
import re
from collections import defaultdict
import torch
from torch_geometric.data import Data

def split_into_chunks(text, max_tokens=50):
    words = text.split()
    chunks = [' '.join(words[i:i + max_tokens]) for i in range(0, len(words), max_tokens)]
    return chunks

def parse_sections(content):
    # Pattern to match any section command, capturing the number of "sub" prefixes and section title
    pattern = r'(\\(sub)*section\{(.*?)\})'
    sections = []
    
    # Find all section commands and their titles
    matches = list(re.finditer(pattern, content))
    for i, match in enumerate(matches):
        section_command = match.group(1)  # Full section command
        title = match.group(3)  # Extract the section title
        start_index = match.start(1)  # Position in the content where the section starts
        end_index = matches[i + 1].start(1) if i + 1 < len(matches) else len(content)
        
        # Determine if this is a leaf node by checking if the next section has a higher or same level
        is_leaf = True  # Default to True (assume it's a leaf)
        if i + 1 < len(matches):
            next_command = matches[i + 1].group(1)
            # It's a leaf only if the next section is of a higher or equal hierarchy
            is_leaf = next_command.count("sub") <= section_command.count("sub")
        
        sections.append({
            "title": title,
            "start_index": start_index,
            "end_index": end_index,
            "is_leaf": is_leaf,
            "depth": section_command.count("sub")  # Depth indicates level of subsection
        })
    
    return sections

def build_graph_from_sections(sections, content, main_title="Main Title", chunk_size=50):
    node_list = []
    edges = []
    node_text = []  # Store text attributes for each node
    section_refs = defaultdict(list)
    section_dict = {}  # Dictionary to map titles to node indices for referencing

    # Add the root node with the main title
    root_node = len(node_list)
    node_list.append({"title": main_title, "content": main_title, "type": "root"})
    node_text.append(main_title)  # Root node text attribute

    # Stack to maintain hierarchy of sections
    stack = [(root_node, -1)]  # (node_index, depth), with root node at depth -1

    for section in sections:
        # Create node for each section title
        section_node = len(node_list)
        node_list.append({"title": section["title"], "content": section["title"], "type": "section"})
        node_text.append(section["title"])  # Add section title to node_text
        section_dict[section["title"]] = section_node  # Map title to node index for cross-references

        # Pop items from the stack until we reach the correct parent level
        while stack and stack[-1][1] >= section["depth"]:
            stack.pop()

        # The top of the stack is the parent node for the current section
        parent_node = stack[-1][0]
        edges.append((parent_node, section_node))

        # Push the current section onto the stack
        stack.append((section_node, section["depth"]))

        # If the section is a leaf, split its content into chunks
        if section["is_leaf"]:
            section_content = content[section["start_index"]:section["end_index"]]
            chunks = split_into_chunks(section_content, max_tokens=chunk_size)

            # Create a node for each chunk and link it to the section
            for chunk in chunks:
                chunk_node = len(node_list)
                node_list.append({"title": "", "content": chunk, "type": "chunk"})
                node_text.append(chunk)
                edges.append((section_node, chunk_node))

                # Check for references in each chunk
                refs = re.findall(r'\\ref\{(.*?)\}', chunk)
                for ref in refs:
                    if ref in section_dict:
                        ref_node = section_dict[ref]
                        edges.append((chunk_node, ref_node))  # Add edge from chunk to referenced section

    return node_list, edges, node_text

def create_pytorch_graph(node_list, edges, node_text):
    # Convert edges to tensor format for PyTorch Geometric
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    num_nodes = len(node_list)
    x = torch.arange(num_nodes, dtype=torch.float32).unsqueeze(1)  # Node feature matrix as indices

    # Create the final PyTorch Geometric Data object
    graph_data = Data(x=x, edge_index=edge_index, node_text=node_text)  # Adding node_text as attribute

    return graph_data

# Example usage
if __name__ == "__main__":
    tex_content = r"""
    \section{Introduction}
    This section provides an overview. As discussed in \ref{related_work}.
    \subsection{Background}
    Background content here.
    \subsubsection{Detailed Background}
    More specific background information.
    \subsubsection{Preliminaries}
    More specific details
    \subsection{Motivation}
    Motivation content. See also \ref{methods}.

    \section{Related Work}
    Related work discussed here.

    \section{Methods}
    Detailed methods are described here.
    """

    # Step 1: Parse sections and identify leaf nodes
    sections = parse_sections(tex_content)

    # Step 2: Build the graph structure with nodes, edges, and node_text
    node_list, edges, node_text = build_graph_from_sections(sections, tex_content, main_title="Sample Paper Title", chunk_size=50)

    # Step 3: Create the PyTorch Geometric graph object
    graph = create_pytorch_graph(node_list, edges, node_text)

    # Output graph data for verification
    print("Node List:", node_list)
    print(f"Edge Index (References):\n{graph.edge_index}")
    print(f"Node Text Attributes:\n{graph.node_text}")
    print(graph)


Node List: [{'title': 'Sample Paper Title', 'content': 'Sample Paper Title', 'type': 'root'}, {'title': 'Introduction', 'content': 'Introduction', 'type': 'section'}, {'title': 'Background', 'content': 'Background', 'type': 'section'}, {'title': 'Detailed Background', 'content': 'Detailed Background', 'type': 'section'}, {'title': '', 'content': '\\subsubsection{Detailed Background} More specific background information.', 'type': 'chunk'}, {'title': 'Preliminaries', 'content': 'Preliminaries', 'type': 'section'}, {'title': '', 'content': '\\subsubsection{Preliminaries} More specific details', 'type': 'chunk'}, {'title': 'Motivation', 'content': 'Motivation', 'type': 'section'}, {'title': '', 'content': '\\subsection{Motivation} Motivation content. See also \\ref{methods}.', 'type': 'chunk'}, {'title': 'Related Work', 'content': 'Related Work', 'type': 'section'}, {'title': '', 'content': '\\section{Related Work} Related work discussed here.', 'type': 'chunk'}, {'title': 'Methods', 'con

In [62]:
import re

def get_section_labels(content):
    # Pattern to match any section command and an optional label
    pattern = r'(\\(sub)*section\{(.*?)\})(?:\s*\\label\{(.*?)\})?'
    section_labels = {}

    # Find all section commands with optional labels
    matches = re.finditer(pattern, content)
    for match in matches:
        section_title = match.group(3)  # Extract the section title
        section_label = match.group(4)  # Extract the optional label, if present
        if section_label:
            section_labels[section_title] = section_label

    return section_labels

# Example usage
if __name__ == "__main__":
    tex_content = r"""
    \section{Introduction}
    This section provides an overview. As discussed in \ref{related_work}.
    \subsection{Background}
    Background content here.
    \subsubsection{Detailed Background}
    More specific background information.
    \subsubsection{Preliminaries}
    More specific details 
    \subsection{Motivation}
    Motivation content. See also \ref{methods}.

    \section{Related Work}\label{related_work}
    Related work discussed here.

    \section{Methods}\label{methods}
    Detailed methods are described here.
    """

    section_labels = get_section_labels(tex_content)
    print("Section Labels Dictionary:", section_labels)


Section Labels Dictionary: {'Related Work': 'related_work', 'Methods': 'methods'}


In [66]:
import re
import torch
from typing import List, Dict, Tuple

def build_reference_edges(node_texts: List[str], edge_index: torch.Tensor, section_labels: Dict[str, str]) -> torch.Tensor:
    # Create reverse mapping from labels to node indices
    label_to_node = {}
    
    # First pass: build mapping of labels to node indices
    for idx, text in enumerate(node_texts):
        # Pattern to match section command with label
        section_pattern = r'\\(?:sub)*section\{(.*?)\}(?:\s*\\label\{(.*?)\})?'
        matches = re.search(section_pattern, text)
        if matches:
            section_title = matches.group(1)
            if section_title in section_labels:
                label = section_labels[section_title]
                label_to_node[label] = idx

    # Convert existing edges to list of tuples
    existing_edges = list(zip(edge_index[0].tolist(), edge_index[1].tolist()))
    all_edges = existing_edges.copy()
    
    # Second pass: find \ref{} commands and add edges
    ref_pattern = r'\\ref\{(.*?)\}'
    for source_idx, text in enumerate(node_texts):
        refs = re.finditer(ref_pattern, text)
        for ref in refs:
            label = ref.group(1)
            if label in label_to_node:
                target_idx = label_to_node[label]
                new_edge = (source_idx, target_idx)
                if new_edge not in all_edges:
                    all_edges.append(new_edge)

    # Convert back to PyTorch tensor
    if all_edges:
        source_nodes = [edge[0] for edge in all_edges]
        target_nodes = [edge[1] for edge in all_edges]
        new_edge_index = torch.tensor([source_nodes, target_nodes], dtype=torch.long)
    else:
        new_edge_index = edge_index.clone()
    
    return new_edge_index

# Example usage:
if __name__ == "__main__":
    # Update graph edge_index
    graph.edge_index = build_reference_edges(graph.node_text, graph.edge_index, section_labels)
    print("Updated edges:", graph.edge_index)

Updated edges: tensor([[ 0,  1,  2,  3,  2,  5,  1,  7,  0,  9,  0, 11,  8],
        [ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 12]])


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from torch_geometric.utils import to_networkx

def visualize_pyg_graph_with_directed_edges(data):
    # Convert the PyTorch Geometric Data object to a NetworkX directed graph
    G = to_networkx(data, to_undirected=False)  # Keep it directed

    # Assign labels to each node from node_text attribute
    labels = {i: data.node_text[i] for i in range(len(data.node_text))}

    # Draw the directed graph with arrows
    pos = nx.spring_layout(G)  # Layout for visualization
    plt.figure(figsize=(12, 8))
    nx.draw(G, pos, with_labels=False, node_size=3000, node_color="skyblue", font_size=10, font_weight="bold", arrows=True)
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=9)
    
    # Add title and show the plot
    plt.title("Directed Graph Visualization with Node Text Attributes")
    plt.show()

visualize_pyg_graph_with_directed_edges(graph)


In [ ]:
# Trun citation graph to hierarchical graph

